In [22]:
import os, sys, socket, torch, pathlib, shutil, subprocess, textwrap
print("cwd:", os.getcwd())
print("host:", socket.gethostname())
print("cuda:", torch.cuda.is_available())
print("content exists:", pathlib.Path("/content").exists())
print("local project exists:", pathlib.Path("/home/yaroslav").exists())
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA version :", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CPU cores visible:", os.cpu_count())

cwd: /content/DUMPLINGs
host: c8b340162eba
cuda: True
content exists: True
local project exists: False
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Executable: /usr/bin/python3
Torch version: 2.10.0+cu128
CUDA version : 12.8
CUDA available: True
GPU name: Tesla T4
CPU cores visible: 2


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
SRC = "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs"
DST = "/content/DUMPLINGs"

os.chdir(SRC)
! git checkout -- config.json
# ! git checkout -- README.md
! git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 592 bytes | 6.00 KiB/s, done.
From https://github.com/BlackSabbitch/DUMPLINGs
   7140840..ace0cd4  main       -> origin/main
Updating 7140840..ace0cd4
Fast-forward
 config.json  |  2 +-
 extractor.py | 10 ++++++++--
 2 files changed, 9 insertions(+), 3 deletions(-)


In [25]:
LOCAL_RUNS = "/content/DUMPLINGs/runs"
DRIVE_RUNS = "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/runs"
os.makedirs(DRIVE_RUNS, exist_ok=True)

if os.path.exists(DST):
    shutil.rmtree(DST)

os.makedirs(DST, exist_ok=True)

for name in [
    "run.py", "trainer.py", "evaluator.py", "extractor.py", "splitter.py",
    "tokenizer.py", "utils.py", "logger.py", "loss_functions.py",
    "config.json", "requirements.txt", "run.sh", "README.md",
    "bad_complexes.toml",
    "models", "parsers", "scripts"
]:
    s = os.path.join(SRC, name)
    d = os.path.join(DST, name)
    if os.path.isdir(s):
        shutil.copytree(s, d)
    elif os.path.exists(s):
        shutil.copy2(s, d)

# ESM cache is optional, but if present we reuse it
esm_cache_src = os.path.join(SRC, "esm_cache")
esm_cache_dst = os.path.join(DST, "esm_cache")
os.makedirs(esm_cache_dst, exist_ok=True)

if os.path.exists(esm_cache_src):
    for name in os.listdir(esm_cache_src):
        s = os.path.join(esm_cache_src, name)
        d = os.path.join(esm_cache_dst, name)
        if os.path.isdir(s):
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
    print(f"Copied ESM cache from {esm_cache_src}")
else:
    print("No ESM cache found on Drive; empty esm_cache/ will be used.")

shutil.copy2(
    "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/pdbbind_v2016.tar.gz",
    "/content/DUMPLINGs/pdbbind_v2016.tar.gz"
)

os.chdir(DST)
print(os.getcwd())

Copied ESM cache from /content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/esm_cache
/content/DUMPLINGs


In [26]:
# Install non-PyG requirements only
skip_prefixes = (
    "torch",
    "torch-geometric",
    "torch-scatter",
    "torch-sparse",
    "torch-cluster",
    "pyg-lib",
)

reqs = []
with open("requirements.txt") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith(skip_prefixes):
            continue
        reqs.append(line)

print("Installing filtered requirements:")
for r in reqs:
    print(" ", r)

subprocess.run(["pip", "install", *reqs], check=True)

Installing filtered requirements:
  rdkit
  biopython
  pandas
  tqdm
  matplotlib
  uniplot
  fair-esm


CompletedProcess(args=['pip', 'install', 'rdkit', 'biopython', 'pandas', 'tqdm', 'matplotlib', 'uniplot', 'fair-esm'], returncode=0)

In [27]:
# Colab environment bootstrap for DUMPLINGs
def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Make matplotlib and temp writes local to the VM, not Drive
os.environ["MPLCONFIGDIR"] = "/content/.mplconfig"
os.environ["TMPDIR"] = "/content/.tmp"
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

# Derive the PyG wheel index from the active torch build
torch_base = torch.__version__.split("+")[0]
cuda_tag = f"cu{torch.version.cuda.replace('.', '')}" if torch.version.cuda else "cpu"
pyg_url = f"https://data.pyg.org/whl/torch-{torch_base}+{cuda_tag}.html"
print("PyG wheel index:", pyg_url)

# Remove possibly incompatible installs
run("pip uninstall -y pyg-lib torch-scatter torch-sparse torch-cluster torch-geometric || true")

# Install the compiled extensions first, then torch-geometric
run(f"pip install pyg-lib torch-scatter torch-sparse torch-cluster -f {pyg_url}")
run("pip install torch-geometric")

# Verify CUDA support in torch_sparse
import torch_sparse
print("torch_sparse:", torch_sparse.__file__)

row = torch.tensor([0, 1], device="cuda")
col = torch.tensor([1, 0], device="cuda")
sp = torch_sparse.SparseTensor(row=row, col=col, sparse_sizes=(2, 2))
print("torch_sparse CUDA check OK:", sp.device())

print(textwrap.dedent("""
Recommended Colab workflow:
1. Keep the repo and training outputs in /content while running.
2. Copy final runs/ and cached artifacts back to Drive after training.
3. Avoid heavy read/write loops directly on mounted Google Drive.
"""))

PyG wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
$ pip uninstall -y pyg-lib torch-scatter torch-sparse torch-cluster torch-geometric || true
$ pip install pyg-lib torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
$ pip install torch-geometric
torch_sparse: /usr/local/lib/python3.12/dist-packages/torch_sparse/__init__.py
torch_sparse CUDA check OK: cuda:0

Recommended Colab workflow:
1. Keep the repo and training outputs in /content while running.
2. Copy final runs/ and cached artifacts back to Drive after training.
3. Avoid heavy read/write loops directly on mounted Google Drive.



In [ ]:
sync_cmd = f"""
while true; do
  rsync -a "{LOCAL_RUNS}/" "{DRIVE_RUNS}/"
  sleep 120
done
"""

sync_proc = subprocess.Popen(["bash", "-lc", sync_cmd])
print("Sync PID:", sync_proc.pid)

Sync PID: 12384


In [29]:
! chmod +x run.sh
! ./run.sh --extract

[INFO][EXPERIMENT] Starting experiment: DUMPLING_A1b_DimeNet_ESM_only_ESM
[INFO][EXPERIMENT] Experiment signature: DUMPLING_A1b_DimeNet_ESM_only_ESM_20260505_114956
[INFO][EXPERIMENT] Base Datasets folder: datasets
[INFO][EXPERIMENT] Base ESM cache folder: esm_cache
[INFO][EXPERIMENT] Run results folder: runs/DUMPLING_A1b_DimeNet_ESM_only_ESM_20260505_114956
[INFO][EXPERIMENT] Log file: runs/DUMPLING_A1b_DimeNet_ESM_only_ESM_20260505_114956/log.txt
[INFO][InteractionGraphParser] Initialized dist_threshold=5.0, ca_only=False
[INFO][CNNParser] Initialized is_ligand=False
[INFO][REGISTRY] Loaded bad complexes registry from bad_complexes.toml with 1 entries
[INFO][EXTRACTION] Unpacking 4057 complexes...
Extracting refined: 66444file [00:56, 1183.26file/s]
[INFO][EXTRACTION] Unpacking completed.
[WARNING][REGISTRY] Excluded 1 complexes from subset refined using bad_complexes.toml: 4bps(training)
[INFO][BUILD] Starting parallel parsing on 2 cores...
100%|██████████| 4056/4056 [02:14<00:00, 3

In [30]:
sync_proc.terminate()

In [31]:
! du -sh /content/DUMPLINGs/esm_cache 2>/dev/null || echo "esm_cache is absent"
! find /content/DUMPLINGs/esm_cache -type f | wc -l
! rsync -a "/content/DUMPLINGs/runs/" "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/runs/"
! rsync -a --delete "/content/DUMPLINGs/esm_cache/" "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/esm_cache/"

20K	/content/DUMPLINGs/esm_cache
2
